# FlexiCache offline top-K profiling — Llama-3.2-1B-Instruct on CoQA

This notebook implements the offline profiling method Nazmul described for FlexiCache
(not included in the released artifact): run vLLM one request at a time, write the
current sample id to a file before each request, and dump the `top_k_blocks` tensor
after every decode step.

## This notebook already fixes 4 known build issues, unconditionally, in order

Every fix below runs automatically every time you run this notebook top to bottom --
none of them are optional or conditional on hitting a specific error message:

1. **Torch/build-isolation mismatch** (`ModuleNotFoundError: No module named 'vllm._C'`) --
   fixed by running `use_existing_torch.py` before the build, so it compiles against
   Kaggle's existing CUDA-linked torch instead of an isolated, mismatched one.
2. **`xgrammar==0.1.16` unavailable on PyPI for this Python/platform** -- patched to
   `xgrammar>=0.1.11` in `requirements/common.txt` before the build starts.
3. **Missing `libcuda.so` symlink** (`CMake Error ... CUDA::cuda_driver ... target was
   not found`) -- common on cloud GPU containers where only the versioned driver lib
   (`libcuda.so.1`) exists. Found and symlinked automatically before the build.
4. **Missing `msgspec` and other runtime deps** -- installed explicitly from
   `requirements/common.txt`/`requirements/cuda.txt` after the build, since the
   editable-install step alone didn't reliably pull in every runtime dependency.

**Target model:** `meta-llama/Llama-3.2-1B-Instruct` (not in FlexiCache's supported list yet --
that's the point: this profiling run is what generates the head-stability data for it).

**GPU:** T4 x2 -- build uses `TORCH_CUDA_ARCH_LIST="7.5"`.

**Steps:**
1. Environment setup + diagnostics
2. Clone FlexiCache and apply patches (model entry, xgrammar pin, profiling hook)
3. Build the patched vLLM from source, with all 4 fixes above applied automatically
4. Verify the build actually produced compiled CUDA extensions before going further
5. Run the CoQA profiling driver (50 samples, batch size 1)
6. Run the existing `TopK-Analysis` stability pipeline on the dumped data

You'll need a Hugging Face token with access to Llama-3.2-1B-Instruct set as the
Kaggle secret `HF_TOKEN` (Add-ons -> Secrets).

**Run this top to bottom in one sitting** (Kernel -> Restart & Run All), rather than
re-running individual cells out of order -- several of the fixes above only take
effect if the earlier setup/clone/patch cells ran first in the same session.

## 1. Environment setup + diagnostics

In [23]:
import os

# T4 has compute capability 7.5. This must match before the from-source build.
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "4"  # Kaggle CPU count is modest; keep parallel build jobs conservative

# Hugging Face auth (needed to download Llama-3.2-1B-Instruct weights)
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!df -h /kaggle/working
!python --version


name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB
Tesla T4, 7.5, 15360 MiB
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.3G   18G  12% /kaggle/working
Python 3.12.13


In [ ]:
# What torch is already on this image, and does it already see the GPU?
# We will build FlexiCache's CUDA kernels AGAINST this existing torch install
# rather than letting pip's build isolation install its own -- see section 3.
import torch
print("pre-installed torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print("torch cuda build version:", torch.version.cuda)


In [ ]:
!python -m pip install -U pip
!python -m pip install "datasets>=2.19" huggingface_hub


## 2. Clone FlexiCache and apply patches

In [ ]:
%cd /kaggle/working
!rm -rf FlexiCache
!git clone https://github.com/NazmulTakbir/FlexiCache.git


In [ ]:
%%writefile /kaggle/working/apply_patches.py
"""
Applies two patches to a freshly cloned NazmulTakbir/FlexiCache checkout:

1. Registers a new model in vllm/v1/flexicache/model_data.py so
   FlexiCacheConfig will accept "meta-llama/Llama-3.2-1B-Instruct" even
   though no pre-computed stable/unstable head profile exists for it yet.
   Run with --num-unstable-heads 0 so FlexiCacheConfig never looks up a
   profile key for this model (see config.py: the model_data lookup is
   skipped entirely when num_unstable_heads == 0).

2. Adds the offline top-K profiling hook described by Nazmul: after every
   decode step, if FLEXICACHE_PROFILE_DIR is set in the environment, read
   the current sample id from FLEXICACHE_SAMPLE_ID_FILE and dump
   input_batch.top_k_blocks[:, i, :, :] to
   {FLEXICACHE_PROFILE_DIR}/{sample_id}/decode-step-{N}-prompt-len-{P}.pt
   This matches exactly what TopK-Analysis/preprocess.py expects to load.

Usage:
    python apply_patches.py /path/to/FlexiCache
"""
import re
import sys
from pathlib import Path


def patch_model_data(repo_root: Path):
    path = repo_root / "vllm" / "v1" / "flexicache" / "model_data.py"
    text = path.read_text()

    marker = "model_data = {"
    if 'meta-llama/Llama-3.2-1B-Instruct' in text:
        print(f"[skip] {path} already patched")
        return

    assert marker in text, f"Could not find insertion point in {path}"
    new_entry = (
        marker
        + '\n    "meta-llama/Llama-3.2-1B-Instruct": {\n'
        + "        # No pre-computed stable/unstable head profile yet.\n"
        + "        # Safe because we always run this model with\n"
        + "        # --num-unstable-heads 0, which skips the profile lookup\n"
        + "        # in FlexiCacheConfig.__init__.\n"
        + "    },"
    )
    text = text.replace(marker, new_entry, 1)
    path.write_text(text)
    print(f"[ok] patched {path}")


def patch_xgrammar_pin(repo_root: Path):
    """
    requirements/common.txt pins xgrammar==0.1.16, which is not published on
    PyPI for some platform/Python combinations (observed: pip resolver finds
    0.1.11, 0.1.13, 0.1.17, ... but not 0.1.16, and aborts the whole install).
    Loosen this to a >=  constraint so pip can pick any available compatible
    build instead of failing dependency resolution before the CUDA extension
    is ever compiled.
    """
    path = repo_root / "requirements" / "common.txt"
    text = path.read_text()

    old_line = 'xgrammar == 0.1.16; platform_machine == "x86_64" or platform_machine == "aarch64"'
    new_line = 'xgrammar >= 0.1.11; platform_machine == "x86_64" or platform_machine == "aarch64"'

    if new_line in text:
        print(f"[skip] {path} already patched")
        return
    assert old_line in text, f"Could not find expected xgrammar line in {path}"
    text = text.replace(old_line, new_line, 1)
    path.write_text(text)
    print(f"[ok] patched {path}")


def patch_gpu_model_runner(repo_root: Path):
    path = repo_root / "vllm" / "v1" / "worker" / "gpu_model_runner.py"
    text = path.read_text()

    if "FLEXICACHE_PROFILE_DIR" in text:
        print(f"[skip] {path} already patched")
        return

    anchor = "                dec_step = new - prompt\n"
    occurrences = text.count(anchor)
    assert occurrences == 1, (
        f"Expected exactly 1 occurrence of anchor line in {path}, found {occurrences}. "
        "The upstream file may have changed; update this patch script."
    )

    hook = anchor + (
        "\n"
        "                # --- offline top-K profiling hook (not part of upstream FlexiCache) ---\n"
        "                _profile_dir = os.environ.get(\"FLEXICACHE_PROFILE_DIR\")\n"
        "                if _profile_dir and dec_step >= 1:\n"
        "                    _sid_file = os.environ.get(\n"
        "                        \"FLEXICACHE_SAMPLE_ID_FILE\",\n"
        "                        \"/tmp/flexicache_current_sample_id.txt\",\n"
        "                    )\n"
        "                    try:\n"
        "                        with open(_sid_file) as _f:\n"
        "                            _sample_id = _f.read().strip()\n"
        "                    except FileNotFoundError:\n"
        "                        _sample_id = None\n"
        "                    if _sample_id:\n"
        "                        _sample_dir = os.path.join(_profile_dir, _sample_id)\n"
        "                        os.makedirs(_sample_dir, exist_ok=True)\n"
        "                        _out_path = os.path.join(\n"
        "                            _sample_dir,\n"
        "                            f\"decode-step-{dec_step}-prompt-len-{prompt}.pt\",\n"
        "                        )\n"
        "                        torch.save(\n"
        "                            self.input_batch.top_k_blocks[:, i, :, :]\n"
        "                            .detach().clone().cpu(),\n"
        "                            _out_path,\n"
        "                        )\n"
        "                # --- end offline top-K profiling hook ---\n"
        "\n"
    )
    text = text.replace(anchor, hook, 1)
    path.write_text(text)
    print(f"[ok] patched {path}")


def main():
    if len(sys.argv) != 2:
        print("usage: python apply_patches.py /path/to/FlexiCache")
        sys.exit(1)
    repo_root = Path(sys.argv[1]).resolve()
    assert (repo_root / "vllm").is_dir(), f"{repo_root} does not look like the FlexiCache repo root"
    patch_model_data(repo_root)
    patch_xgrammar_pin(repo_root)
    patch_gpu_model_runner(repo_root)
    print("Done.")


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/apply_patches.py /kaggle/working/FlexiCache


In [ ]:
# Sanity-check the patches landed where expected before kicking off the build.
!grep -n "Llama-3.2-1B-Instruct" /kaggle/working/FlexiCache/vllm/v1/flexicache/model_data.py
!grep -n "xgrammar" /kaggle/working/FlexiCache/requirements/common.txt
!grep -n "FLEXICACHE_PROFILE_DIR" /kaggle/working/FlexiCache/vllm/v1/worker/gpu_model_runner.py | head -5
!python -c "import ast; ast.parse(open('/kaggle/working/FlexiCache/vllm/v1/worker/gpu_model_runner.py').read()); print('gpu_model_runner.py OK')"


## 3. Build the patched vLLM from source

**This is the section that broke previously.** The root cause: `pip install -e .`
defaults to *build isolation* -- it creates a throwaway virtual env, installs the
`torch==2.6.0` pinned in `requirements/build.txt` into it (a plain CPU/generic wheel,
not linked against Kaggle's actual CUDA driver), and compiles the custom CUDA
extension (`vllm._C`) against *that* torch instead of the working, CUDA-linked
torch Kaggle already gave you. The compile step then either fails quietly or
produces an extension that doesn't match your runtime -- which is exactly the
`ModuleNotFoundError: No module named 'vllm._C'` you hit.

The fix is a script the upstream vLLM project ships for exactly this situation
(pre-installed, CUDA-linked torch, e.g. Colab/Kaggle-style images):
`use_existing_torch.py`. It strips the `torch` version pin out of every
requirements file, so the build uses whatever torch is already on the box instead
of trying to install its own. We then install with `--no-build-isolation` so pip
doesn't sandbox the build away from that existing torch at all.

On T4/P100, `TORCH_CUDA_ARCH_LIST="7.5"` only controls which GPU architecture's
machine code gets compiled (faster build, not a Hopper hard-dependency -- see
Nazmul's email) -- Triton kernel launch configs (block size/warps/stages) tuned
for H100 may just not be optimal here, which is fine for a profiling run.

This step can take 20-40 minutes. We tee the **full** log to a file this time --
past attempts lost the real error by only looking at `tail -80`.

In [ ]:
%cd /kaggle/working/FlexiCache

# Use whatever torch Kaggle already installed instead of letting the build
# reinstall its own (mismatched, non-CUDA-linked) copy.
!python use_existing_torch.py


In [ ]:
# Build-time tooling (cmake/ninja) -- installed via pip per requirements/build.txt,
# with the torch pin already stripped by use_existing_torch.py above.
!pip install -r requirements/build.txt 2>&1 | tail -40


In [ ]:
# Belt-and-suspenders: make sure the system has a C/C++ toolchain and cmake/ninja
# available even if the pip-installed versions have issues.
!apt-get update -qq && apt-get install -y -qq build-essential cmake ninja-build > /dev/null
!gcc --version | head -1
!cmake --version | head -1
!ninja --version


In [ ]:
# Belt-and-suspenders check, independent of apply_patches.py having run
# correctly earlier in this session: read the file fresh, patch it if
# needed, and HARD-FAIL here (before the 30-min build) if the bad pin is
# still present, rather than discovering it in the build log again.
req_path = "/kaggle/working/FlexiCache/requirements/common.txt"
with open(req_path) as f:
    content = f.read()

before = [l for l in content.splitlines() if "xgrammar" in l]
print("xgrammar line(s) BEFORE:", before)

if "xgrammar == 0.1.16" in content:
    content = content.replace(
        'xgrammar == 0.1.16; platform_machine == "x86_64" or platform_machine == "aarch64"',
        'xgrammar >= 0.1.11; platform_machine == "x86_64" or platform_machine == "aarch64"',
    )
    with open(req_path, "w") as f:
        f.write(content)

with open(req_path) as f:
    content = f.read()
after = [l for l in content.splitlines() if "xgrammar" in l]
print("xgrammar line(s) AFTER: ", after)

assert "0.1.16" not in content, (
    "xgrammar==0.1.16 pin is still present -- something is re-cloning or "
    "reverting this file. Check you're not re-running the git clone cell "
    "after this point without re-running this fix."
)


**Fix #3 (libcuda.so symlink) -- corrected version.**

An earlier version of this fix only set `LIBRARY_PATH`, which affects the *linker*
at actual link time -- but CMake's `find_package(CUDAToolkit)` builds the
`CUDA::cuda_driver` target during the **configure** step using its own search
logic, which specifically looks inside the CUDA toolkit's own `lib64/stubs`
(or `lib/stubs`) directory -- not wherever the system's real runtime driver
library happens to live. That stubs directory is where NVIDIA's toolkit
normally ships a placeholder `libcuda.so` for exactly this build-without-a-
full-driver scenario; on a pip-installed CUDA toolkit (common on Kaggle) it
may not exist at all. This version finds every plausible toolkit root and
creates the symlink in the actual location CMake looks for it.

In [ ]:
"""
Corrected fix for:
  CMake Error ... target_link_libraries: Target "_C" links to: CUDA::cuda_driver
  but the target was not found.

The earlier version of this fix only set LIBRARY_PATH, which affects the
*linker* at actual link time -- but CMake's find_package(CUDAToolkit) builds
the CUDA::cuda_driver imported target during the CONFIGURE step using its own
search logic, which specifically looks inside the CUDA toolkit installation's
own lib64/stubs (or lib/stubs) directory -- NOT wherever the system's real
runtime driver library happens to live. That stubs directory is where NVIDIA's
toolkit normally ships a placeholder libcuda.so precisely so you can build
CUDA code on a machine without a full driver install. If the CUDA toolkit here
was installed via pip wheels (common on Kaggle) rather than a full system
install, that stubs directory may not exist at all.

This finds every plausible CUDA toolkit root, creates a lib64/stubs (and
lib/stubs) directory under each if missing, and symlinks the real driver
library into it -- which is where CMake's FindCUDAToolkit module actually
looks for this specific target.
"""
import glob
import os
import subprocess


def find_real_libcuda():
    candidates = []
    for pattern in ("/usr/lib/**/libcuda.so*", "/usr/lib64/**/libcuda.so*",
                     "/usr/local/nvidia/lib64/libcuda.so*"):
        candidates += glob.glob(pattern, recursive=True)
    candidates = [c for c in candidates if not c.endswith("libcuda.so")]  # prefer the versioned real file
    if not candidates:
        # fall back to including any match at all, even an existing unversioned one
        for pattern in ("/usr/lib/**/libcuda.so*", "/usr/lib64/**/libcuda.so*"):
            candidates += glob.glob(pattern, recursive=True)
    assert candidates, "No libcuda.so* found anywhere -- run `!find / -name 'libcuda.so*' 2>/dev/null` manually"
    return sorted(set(candidates), key=len)[-1]


def find_cuda_toolkit_roots():
    roots = set()

    # 1. Standard system install locations
    for p in glob.glob("/usr/local/cuda*"):
        if os.path.isdir(p):
            roots.add(p)

    # 2. nvcc on PATH -> toolkit root is typically two levels up from its bin/
    nvcc = subprocess.run(["which", "nvcc"], capture_output=True, text=True).stdout.strip()
    if nvcc:
        roots.add(os.path.dirname(os.path.dirname(nvcc)))

    # 3. pip-installed nvidia-* CUDA packages (common on Kaggle/Colab-style images)
    try:
        import nvidia
        nvidia_root = os.path.dirname(nvidia.__file__)
        for sub in os.listdir(nvidia_root):
            sub_path = os.path.join(nvidia_root, sub)
            if os.path.isdir(sub_path) and ("cuda" in sub.lower() or "nvrtc" in sub.lower()):
                roots.add(sub_path)
    except ImportError:
        pass

    return sorted(r for r in roots if os.path.isdir(r))


def main():
    real_lib = find_real_libcuda()
    print(f"Real driver library: {real_lib}")

    roots = find_cuda_toolkit_roots()
    print(f"\nCandidate CUDA toolkit roots:")
    for r in roots:
        print(" ", r)

    if not roots:
        print("\nNo CUDA toolkit root found via the usual methods.")
        print("Run `!python -c \"import torch; print(torch.utils.cpp_extension.CUDA_HOME)\"`")
        print("and tell me what it prints so I can target the right directory.")
        return

    created_any = False
    for root in roots:
        for stub_subdir in ("lib64/stubs", "lib/stubs"):
            stub_dir = os.path.join(root, stub_subdir)
            link_path = os.path.join(stub_dir, "libcuda.so")
            try:
                os.makedirs(stub_dir, exist_ok=True)
                if not os.path.exists(link_path):
                    os.symlink(real_lib, link_path)
                    print(f"Created: {link_path} -> {real_lib}")
                    created_any = True
                else:
                    print(f"Already exists: {link_path}")
                    created_any = True
            except (PermissionError, OSError) as e:
                print(f"Could not create {link_path}: {e}")

    if not created_any:
        print("\nCould not create the stub symlink anywhere -- likely a permissions "
              "issue. Try prefixing the failing commands with sudo, or tell me the "
              "error above and I'll adjust.")
    else:
        print("\nDone. Re-run:  !rm -rf /kaggle/working/FlexiCache/build")
        print("Then re-run the build cell.")


if __name__ == "__main__":
    main()


**Fix #5 (stale CMake cache) -- clears any cached build state from a previous failed attempt.** CMake remembers whether it found the CUDA driver library the first time it ran; if that check failed before the libcuda.so symlink existed, it can keep replaying that cached failure even after the symlink fix above runs. Deleting the build directory forces a fresh check every time.

In [ ]:
!rm -rf /kaggle/working/FlexiCache/build


In [ ]:
# The actual build. --no-build-isolation is the important flag: it tells pip to
# build directly in THIS environment (with the existing, CUDA-linked torch)
# instead of an isolated one. Full output goes to a file so we can search it below
# instead of losing the real error to a truncated tail.
!pip install -e . --no-build-isolation -v 2>&1 | tee /kaggle/working/build_log.txt


**Stop and check here before continuing.** Don't just assume it worked --
verify the compiled extension actually exists.

In [ ]:
import subprocess, os

so_files = subprocess.run(
    ["find", "/kaggle/working/FlexiCache", "-name", "_C*.so"],
    capture_output=True, text=True
).stdout.strip()

if so_files:
    print("Found compiled extension(s):")
    print(so_files)
else:
    print("!!! No _C*.so found -- the CUDA extension did NOT build. !!!")
    print("Searching build_log.txt for the real error...\n")
    with open("/kaggle/working/build_log.txt") as f:
        lines = f.readlines()
    hits = [
        (i, l) for i, l in enumerate(lines)
        if any(k in l.lower() for k in ("error", "fail", "traceback", "no space", "killed"))
    ]
    for i, l in hits[-60:]:
        print(f"{i}: {l.rstrip()}")
    raise RuntimeError(
        "Build did not produce vllm._C -- inspect the errors printed above "
        "(or open /kaggle/working/build_log.txt in full) before proceeding."
    )


In [ ]:
# Now the rest of FlexiCache's own runtime requirements (also torch-pin-free now).
!pip install -r requirements/common.txt 2>&1 | tail -40
!pip install -r requirements/cuda.txt 2>&1 | tail -40
!pip install -r flexicache_requirements.txt 2>&1 | tail -40


In [ ]:
import importlib
import vllm
importlib.reload(vllm)
print("vllm:", vllm.__version__)

import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

import vllm._C  # noqa -- this is the import that failed before; if this line
                # succeeds, the build is genuinely complete.
print("vllm._C imported successfully")


In [ ]:
!pip install --force-reinstall --no-cache-dir "numpy<2.0.0"

## 4. Run the CoQA profiling driver

In [24]:
%%writefile /kaggle/working/profile_coqa.py
"""
Offline top-K profiling driver for Llama-3.2-1B-Instruct on CoQA.

Mirrors the one-off instrumentation approach described by Nazmul:
  - batch size 1, requests sent to vLLM one at a time
  - before each request, write the current sample id to a file
  - the patched gpu_model_runner.py reads that file and, after every
    decode step, dumps the current top_k_blocks tensor to disk

Run with FLEXICACHE_PROFILE_DIR already exported (the notebook does this).
"""
import os
import re
import json
from pathlib import Path

from datasets import load_dataset
from vllm import LLM, SamplingParams

MODEL = "meta-llama/Llama-3.2-1B-Instruct"

# TopK-Analysis/preprocess.py's list_samples() expects:
#   data_root/<model_dirname>/<dataset_name>/sample-XXXX/decode-step-*.pt
# so PROFILE_DIR must already be "<data_root>/<model_dirname>/<dataset_name>"
# -- see DATA_ROOT / MODEL_DIRNAME below, set from the notebook.
DATA_ROOT = os.environ.get("FLEXICACHE_DATA_ROOT", "/kaggle/working/topk_profile_data")
MODEL_DIRNAME = os.environ.get("FLEXICACHE_MODEL_DIRNAME", "Llama-3.2-1B-Instruct")
DATASET_NAME = os.environ.get("FLEXICACHE_DATASET_NAME", "coqa")
PROFILE_DIR = os.environ.get(
    "FLEXICACHE_PROFILE_DIR", os.path.join(DATA_ROOT, MODEL_DIRNAME, DATASET_NAME)
)
os.environ["FLEXICACHE_PROFILE_DIR"] = PROFILE_DIR  # so the patched worker sees it
SAMPLE_ID_FILE = os.environ.get(
    "FLEXICACHE_SAMPLE_ID_FILE", "/tmp/flexicache_current_sample_id.txt"
)
N_SAMPLES = int(os.environ.get("FLEXICACHE_N_SAMPLES", "50"))
BLOCK_SIZE = int(os.environ.get("FLEXICACHE_BLOCK_SIZE", "16"))
TOPK_BUDGET = int(os.environ.get("FLEXICACHE_TOPK_BUDGET", "64"))
RERANK_FREQUENCY = int(os.environ.get("FLEXICACHE_RERANK_FREQUENCY", "16"))
MAX_MODEL_LEN = int(os.environ.get("FLEXICACHE_MAX_MODEL_LEN", "4096"))
MAX_NEW_TOKENS = int(os.environ.get("FLEXICACHE_MAX_NEW_TOKENS", "64"))


def build_coqa_prompt(story: str, questions, answers, turn_idx: int) -> str:
    """
    CoQA is a multi-turn conversational-QA dataset: one passage ("story"),
    followed by a sequence of questions, each answered using the story and
    the preceding QA turns as context. We linearize turns [0, turn_idx]
    into a single prompt and ask the model to answer question turn_idx.
    """
    lines = [
        "Answer the final question using only the passage and the "
        "conversation so far. Be concise.",
        "",
        "Passage:",
        story.strip(),
        "",
        "Conversation:",
    ]
    for t in range(turn_idx):
        lines.append(f"Q: {questions[t]}")
        lines.append(f"A: {answers[t]}")
    lines.append(f"Q: {questions[turn_idx]}")
    lines.append("A:")
    return "\n".join(lines)


def iter_coqa_examples(n_samples: int):
    """
    Yields (sample_id, prompt) pairs. Each CoQA "story" has multiple
    question turns; we take the first turn of each of the first
    n_samples stories to keep sample_ids simple and 1:1 with prompts.
    Adjust here if you'd rather profile every turn of every story.
    """
    ds = load_dataset("stanfordnlp/coqa", split="validation")
    count = 0
    for row in ds:
        if count >= n_samples:
            break
        story = row["story"]
        questions = row["questions"]
        answers = row["answers"]["input_text"]
        if not questions:
            continue
        prompt = build_coqa_prompt(story, questions, answers, turn_idx=0)
        sample_id = f"sample-{count:04d}"
        yield sample_id, prompt
        count += 1


def write_sample_id(sample_id: str) -> None:
    Path(SAMPLE_ID_FILE).write_text(sample_id)


def main():
    os.makedirs(PROFILE_DIR, exist_ok=True)
    Path(SAMPLE_ID_FILE).touch()

    llm = LLM(
        model=MODEL,
        enforce_eager=True,          # simpler/safer for a one-off profiling run
        gpu_memory_utilization=0.85,
        max_model_len=MAX_MODEL_LEN,
        block_size=BLOCK_SIZE,
        enable_flexicache=True,
        num_unstable_heads=0,        # no profile needed yet -- that's what we're building
        rerank_frequency=RERANK_FREQUENCY,
        topK_budget=TOPK_BUDGET,
        unstable_heads_profile_task="none",
        enable_prefix_caching=False,
        disable_cascade_attn=True,
    )
    sampling_params = SamplingParams(max_tokens=MAX_NEW_TOKENS, temperature=0.0)

    manifest = []
    for sample_id, prompt in iter_coqa_examples(N_SAMPLES):
        write_sample_id(sample_id)
        outputs = llm.generate([prompt], sampling_params, use_tqdm=False)
        generated = outputs[0].outputs[0].text
        manifest.append({"sample_id": sample_id, "generated": generated})
        print(f"[{sample_id}] {generated[:80]!r}")

    # Clear the sample id file so nothing writes profiling data after we're done.
    Path(SAMPLE_ID_FILE).write_text("")

    # NOTE: write the manifest at DATA_ROOT, not inside PROFILE_DIR --
    # list_samples() in TopK-Analysis/preprocess.py assumes every entry
    # under data_root/model/dataset is a "sample-*" directory, so a stray
    # file inside PROFILE_DIR would break it.
    with open(os.path.join(DATA_ROOT, f"manifest_{MODEL_DIRNAME}_{DATASET_NAME}.jsonl"), "w") as f:
        for row in manifest:
            f.write(json.dumps(row) + "\n")

    print(f"\nDone. Profiled {len(manifest)} samples into {PROFILE_DIR}")


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/profile_coqa.py


In [25]:
import os

os.environ["FLEXICACHE_DATA_ROOT"] = "/kaggle/working/topk_profile_data"
os.environ["FLEXICACHE_MODEL_DIRNAME"] = "Llama-3.2-1B-Instruct"
os.environ["FLEXICACHE_DATASET_NAME"] = "coqa"
os.environ["FLEXICACHE_SAMPLE_ID_FILE"] = "/kaggle/working/.flexicache_current_sample_id"
os.environ["FLEXICACHE_N_SAMPLES"] = "50"
os.environ["FLEXICACHE_BLOCK_SIZE"] = "16"
os.environ["FLEXICACHE_TOPK_BUDGET"] = "64"
os.environ["FLEXICACHE_RERANK_FREQUENCY"] = "16"
os.environ["FLEXICACHE_MAX_MODEL_LEN"] = "4096"
os.environ["FLEXICACHE_MAX_NEW_TOKENS"] = "64"

os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN_VLLM_V1"
os.environ["VLLM_USE_V1"] = "1"


In [26]:
%cd /kaggle/working
!python profile_coqa.py


/kaggle/working
INFO 09-19 08:49:25 [__init__.py:239] Automatically detected platform cuda.
config.json: 100%|█████████████████████████████| 877/877 [00:00<00:00, 4.44MB/s]
INFO 09-19 08:49:48 [config.py:587] This model supports multiple tasks: {'embed', 'score', 'reward', 'classify', 'generate'}. Defaulting to 'generate'.
Traceback (most recent call last):
  File "/kaggle/working/profile_coqa.py", line 138, in <module>
    main()
  File "/kaggle/working/profile_coqa.py", line 99, in main
    llm = LLM(
          ^^^^
  File "/kaggle/working/FlexiCache/vllm/utils.py", line 1037, in inner
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/FlexiCache/vllm/entrypoints/llm.py", line 258, in __init__
    self.llm_engine = LLMEngine.from_engine_args(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/FlexiCache/vllm/v1/engine/llm_engine.py", line 131, in from_engine_args
    vllm_config = engine_args.create_engine_config(usage_context

In [6]:
# Quick check of what got written
import os
data_root = "/kaggle/working/topk_profile_data/Llama-3.2-1B-Instruct/coqa"
samples = sorted(os.listdir(data_root))
print(f"{len(samples)} sample dirs")
print(samples[:5])

first = samples[0]
files = sorted(os.listdir(os.path.join(data_root, first)))
print(f"\n{first}: {len(files)} decode-step dumps")
print(files[:5])

import torch
t = torch.load(os.path.join(data_root, first, files[0]))
print(f"\ntensor shape [L, H, K] = {tuple(t.shape)}, dtype={t.dtype}")


0 sample dirs
[]


IndexError: list index out of range

## 5. Run the existing stability analysis on the profiled data

`TopK-Analysis/analyze_head_stability.py` already implements the RCO-based stability
metric from the paper (`build_topk_masks_all_heads`, `compute_chunk_instability`,
`get_unstable_heads_per_chunk`) — we're just pointing it at the data we just
generated instead of the paper's own dumps.

In [ ]:
%cd /kaggle/working/FlexiCache/TopK-Analysis
import sys
sys.path.insert(0, ".")
from analyze_head_stability import analyze_model

summary = analyze_model(
    data_root="/kaggle/working/topk_profile_data",
    model="Llama-3.2-1B-Instruct",
    out_root="/kaggle/working/stability_analysis_llama32_1b_coqa",
    window_len=16,      # matches --rerank-frequency used during profiling
    Ms=[64],            # top-K page budget(s) to analyze, matches --topK-budget
    page_size=16,       # matches --block_size used during profiling
    topK=64,
)
print(summary)


In [1]:
import vllm
print(vllm.__version__)
import vllm._C
print("vllm._C imported successfully")

INFO 09-19 08:39:59 [__init__.py:239] Automatically detected platform cuda.
3.dev1+gab0f495b1.d20260919
vllm._C imported successfully


In [22]:
import os
print(os.environ.get("HF_TOKEN"))

None


`summary` and the per-dataset outputs under
`/kaggle/working/stability_analysis_llama32_1b_coqa/` give you, per (layer, head), the
instability score (1 - mean RCO). From there you can pick `num_unstable_heads` and
build a `model_data.py` entry the same shape as the ones already in the repo for
Llama-3.1-8B-Instruct / Mistral, then re-run with `--enable-flexicache
--num-unstable-heads <N> --unstable-heads-profile-task coqa` for real.